In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
! apt-get update && apt-get install -y build-essential libc6-dev python3-dev

In [ ]:
!find /usr -name "libcuda.so*"

In [1]:
%%bash
# 1. Tạo thư mục stubs riêng có quyền ghi (tránh lỗi Read-only)
mkdir -p /tmp/cuda_stubs

# 2. Tạo symlink trỏ tới file libcuda.so thực tế
ln -sf /usr/local/nvidia/lib64/libcuda.so /tmp/cuda_stubs/libcuda.so

# 3. Export biến môi trường trỏ vào thư mục stubs vừa tạo
export CUDA_HOME=/usr/local/cuda
export LIBRARY_PATH=/tmp/cuda_stubs:$LIBRARY_PATH
export LD_LIBRARY_PATH=/tmp/cuda_stubs:/usr/local/nvidia/lib64:/usr/local/cuda-12.8/compat:$LD_LIBRARY_PATH

# 4. Kiểm tra lại ngay lập tức
c++ -print-file-name=libcuda.so

/tmp/cuda_stubs/libcuda.so


In [ ]:
%cd /kaggle/working
!git clone https://github.com/Djuybu/r2AI_2026


In [ ]:
%cd /kaggle/working/r2AI_2026

In [ ]:
!pip install -q \
    "protobuf>=5.29.6,<6.0dev" \
    "starlette>=0.40.0,<1.0.0" \
    "opentelemetry-api>=1.35.0,<1.39.0" \
    "opentelemetry-sdk>=1.35.0,<1.39.0" \
    "numba>=0.60.0,<0.63.0" \
    "google-cloud-bigquery-storage>=2.30.0,<3.0.0" \
    "langgraph>=0.2.0" \
    "langchain-core>=0.3.0" \
    "langchain-openai>=0.2.0" \
    "vllm>=0.6.0" \
    "pyyaml>=6.0" \
    "json-repair>=0.30.0" \
    "openpyxl>=3.1.0" \
    "flashinfer-python" \
    "tabulate>=0.9.0" \
    "thefuzz>=0.22.0"

In [3]:
import os
import sys

# ==========================================
# 0. CẤU HÌNH CUDA STUBS (TRÁNH LỖI NINJA BUILD)
# ==========================================
import subprocess
os.makedirs("/tmp/cuda_stubs", exist_ok=True)
if not os.path.exists("/tmp/cuda_stubs/libcuda.so"):
    if os.path.exists("/usr/local/nvidia/lib64/libcuda.so"):
        subprocess.run(["ln", "-sf", "/usr/local/nvidia/lib64/libcuda.so", "/tmp/cuda_stubs/libcuda.so"])

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["LIBRARY_PATH"] = f"/tmp/cuda_stubs:{os.environ.get('LIBRARY_PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"/tmp/cuda_stubs:/usr/local/nvidia/lib64:/usr/local/cuda-12.8/compat:{os.environ.get('LD_LIBRARY_PATH', '')}"

# ==========================================
# 1. CẤU HÌNH MÔI TRƯỜNG KAGGLE DUAL GPU
# ==========================================
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["OMP_NUM_THREADS"] = "4"

from kaggle_secrets import UserSecretsClient

# ==========================================
# 2. TỰ ĐỘNG QUÉT MODEL TỪ KAGGLE INPUT
# ==========================================
def get_model_path(default_hf_id="Qwen/Qwen3.5-4B"):
    input_dir = "/kaggle/input"
    if os.path.exists(input_dir):
        for root, dirs, files in os.walk(input_dir):
            if "config.json" in files or any(f.endswith(".safetensors") for f in files):
                print(f"📁 Đã tìm thấy local model tại Kaggle Input: {root}")
                return root
    return default_hf_id

MODEL_ID = get_model_path("Qwen/Qwen3.5-4B")

# ==========================================
# 3. KHỞI TẠO VLLM PYTHON NATIVE (DUAL GPU)
# ==========================================
from vllm import LLM, SamplingParams

print(f"🚀 Khởi tạo vLLM Native Python Engine trên 2 GPU T4 (Model: {MODEL_ID})...")

llm = LLM(
    model=MODEL_ID,
    tensor_parallel_size=2,        # Chia đều model cho 2 GPU T4
    dtype="float16",               # T4 float16
    gpu_memory_utilization=0.75,   # Giữ VRAM chừa bộ nhớ PyTorch IPC
    max_model_len=2048,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    trust_remote_code=True
)

print("✅ Khởi tạo vLLM Native Engine 2 GPU THÀNH CÔNG!")

📁 Đã tìm thấy local model tại Kaggle Input: /kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1
🚀 Khởi tạo vLLM Native Python Engine trên 2 GPU T4 (Model: /kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1)...
INFO 08-09 06:24:34 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'float16', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'model': '/kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1'}
INFO 08-09 06:24:34 [model.py:549] Resolved architecture: Qwen3ForCausalLM
WARNING 08-09 06:24:34 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 08-09 06:24:34 [model.py:1678] Using max model len 2048
WARNING 08-09 06:24:34 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 08-09 06:24:34 [vllm.py:859] Inductor compilation was disabled 

(Worker pid=1339) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=1338) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=1339) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=1338) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(Worker pid=1338) INFO 08-09 06:25:11 [pynccl.py:111] vLLM is using nccl==2.27.5
(Worker pid=1338) WARNING 08-09 06:25:11 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=1339) WARNING 08-09 06:25:11 [symm_mem.py:66] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=1338) INFO 08-09 06:25:11 [parallel_state.py:1716] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(Worker_TP0 pid=1338) INFO 08-09 06:25:12 [gpu_model_runner.py:4735] Starting to load model /kaggle/input/models/qwen-lm/qwen-3/transformers/4b/1...
(Worker_TP0 pid=1338) ERROR 08-09 06:25:12 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
(Worker_TP1 pid=1339) ERROR 08-09 06:25:12 [fa_utils.py:145] Cannot use FA version 2 is not supported due to FA2 is only supported o

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


(Worker_TP1 pid=1339) INFO 08-09 06:25:13 [weight_utils.py:848] Prefetching checkpoint files into page cache started (in background)
(Worker_TP0 pid=1338) INFO 08-09 06:25:13 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/2)


Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:11<00:22, 11.37s/it]


(Worker_TP0 pid=1338) INFO 08-09 06:25:24 [weight_utils.py:825] Prefetching checkpoint files: 20% (2/2)
(Worker_TP0 pid=1338) INFO 08-09 06:25:24 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 11.48s


Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:17<00:08,  8.36s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:17<00:00,  5.90s/it]
(Worker_TP0 pid=1338) 


(Worker_TP1 pid=1339) INFO 08-09 06:25:30 [weight_utils.py:825] Prefetching checkpoint files: 10% (1/1)
(Worker_TP1 pid=1339) INFO 08-09 06:25:30 [weight_utils.py:843] Prefetching checkpoint files into page cache finished in 17.54s
(Worker_TP0 pid=1338) INFO 08-09 06:25:30 [default_loader.py:384] Loading weights took 17.72 seconds
(Worker_TP0 pid=1338) INFO 08-09 06:25:31 [gpu_model_runner.py:4820] Model loading took 3.82 GiB memory and 18.088701 seconds
(Worker_TP0 pid=1338) INFO 08-09 06:25:51 [gpu_worker.py:436] Available KV cache memory: 6.49 GiB
(EngineCore pid=1315) INFO 08-09 06:25:51 [kv_cache_utils.py:1319] GPU KV cache size: 94,544 tokens
(EngineCore pid=1315) INFO 08-09 06:25:51 [kv_cache_utils.py:1324] Maximum concurrency for 2,048 tokens per request: 46.16x
(Worker_TP0 pid=1338) INFO 08-09 06:25:51 [kernel_warmup.py:69] Warming up FlashInfer attention.
(Worker_TP1 pid=1339) INFO 08-09 06:25:51 [kernel_warmup.py:69] Warming up FlashInfer attention.
(EngineCore pid=1315) INF

In [ ]:
# Cell kiểm tra nguyên nhân gốc bị ẩn trong vllm_stderr.log
with open("vllm_stderr.log", "r") as f:
    text = f.read()

print("=== DÒNG BÁO LỖI GỐC (ROOT CAUSE) ===")
for line in text.split("\n"):
    # In ra các dòng chứa từ khóa báo lỗi của Worker
    if any(k in line for k in ["Error", "Exception", "Failed", "failed", "cannot", "NotFoundError", "OOM"]):
        print(line)

In [4]:
# 4. TEST SỬ DỤNG TRỰC TIẾP VLLM PYTHON API
prompts = ["Xin chào! Bạn có thể giúp gì cho tôi trong việc phân tích dữ liệu Pandas?"]
sampling_params = SamplingParams(temperature=0.0, max_tokens=150)

outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    print("🤖 Output từ 2 GPU:", output.outputs[0].text)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

🤖 Output từ 2 GPU:  Xin lỗi, tôi không thể hỗ trợ trực tiếp trong việc phân tích dữ liệu Pandas. Tuy nhiên, tôi có thể giúp bạn hiểu về Pandas và cung cấp hướng dẫn để bạn tự thực hiện phân tích dữ liệu. Bạn có thể mô tả cụ thể hơn về vấn đề bạn đang gặp phải hoặc mục tiêu phân tích của bạn? Tôi có thể giúp bạn giải thích các hàm, phương pháp hoặc kỹ thuật liên quan đến Pandas. Ví dụ, bạn có thể cần giúp với việc xử lý dữ liệu, phân tích thống kê, vẽ biểu đồ, hoặc điều chỉnh dữ liệu? Xin lỗi, tôi không thể hỗ trợ trực tiếp trong việc phân tích dữ liệu Pandas. Tuy nhiên, tôi có


In [5]:
# 5. TẠO OPENAI-COMPATIBLE API SERVER TRONG BACKGROUND THREAD (CHO LANGCHAIN)
import threading
from fastapi import FastAPI
import uvicorn
from pydantic import BaseModel
from typing import List, Optional

app = FastAPI()

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model: str
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.0
    max_tokens: Optional[int] = 512

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    prompt = "\n".join([f"{m.role}: {m.content}" for m in req.messages])
    sp = SamplingParams(temperature=req.temperature, max_tokens=req.max_tokens)
    res = llm.generate([prompt], sp)
    text = res[0].outputs[0].text
    return {
        "id": "chatcmpl-cocopila",
        "object": "chat.completion",
        "model": req.model,
        "choices": [{
            "index": 0,
            "message": {"role": "assistant", "content": text},
            "finish_reason": "stop"
        }]
    }

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🌐 OpenAI API Server sẵn sàng tại http://localhost:8000/v1 !")

🌐 OpenAI API Server sẵn sàng tại http://localhost:8000/v1 !


In [6]:
# 6. TEST LANGCHAIN VỚI OPENAI API PORT 8000
import sys
sys.path.insert(0, "/kaggle/working/r2AI_2026")

from langchain_openai import ChatOpenAI

langchain_llm = ChatOpenAI(
    model=MODEL_ID,
    openai_api_base="http://localhost:8000/v1",
    openai_api_key="EMPTY",
    temperature=0,
)

res = langchain_llm.invoke("Xin chào! Kiểm tra kết nối từ LangChain?")
print("🤖 Phản hồi từ LangChain:", res.content)

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

🤖 Phản hồi từ LangChain:  Tôi đang gặp một lỗi khi chạy một ứng dụng với LangChain. Lỗi là: "ValueError: The 'llm' parameter is required for the 'llm' type. Please provide a valid LLM model or a function that returns a valid LLM model."

Tôi đang sử dụng một mô hình LLM như là một phần của một ứng dụng web. Tôi đã cài đặt các thư viện cần thiết, nhưng vẫn không thể chạy ứng dụng. Tôi cần giúp đỡ để xác định nguyên nhân và cách khắc phục.

Tôi đang sử dụng Python 3.11.5, và các thư viện như langchain, langchain-community, langchain-llms, langchain-core, langchain-prompt, langchain-text-splitters, langchain-chains, langchain-agents, langchain-llm, langchain-embeddings, langchain-verbose, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, langchain-llms, l